# 层和块
## 自定义块

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [34]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()# 初始化所有父类（Module）的属性，方便之后定义weight，bias
        # 定义网络结构
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)
    
    def forward(self, X):
        return self.out(F.relu(self.hidden(X))) # 定义前向传播的输出，调用定义的层

### 实例化多层感知机的层

In [35]:
net = MLP() # 实例化网络
X = torch.randn(2,20) # 生成一个2行20列的随机张量
net(X) # 前向传播，输出一个2行10列的张量

tensor([[ 0.1188,  0.7094, -0.3066, -0.0889, -0.3487, -0.3653,  0.2684, -0.3602,
          0.1144,  0.0541],
        [ 0.0842,  0.0086, -0.4503, -0.0761, -0.1414,  0.0502, -0.1208, -0.4605,
          0.0034,  0.3085]], grad_fn=<AddmmBackward0>)

## 顺序块

In [6]:
class My_sequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for block in args:
            self._modules[block] = block  # 将每个block添加到_module字典中

    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X
    
# 实例化My_sequential并传入MLP实例
net = My_sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10)) # 参数传入__init__方法的args中
X = torch.randn(2, 20)  # 生成一个2行20列的随机张量
output = net(X) # X传入forward方法
output

tensor([[-0.2693, -0.0343,  0.2339,  0.0957, -0.3693, -0.0426,  0.2030, -0.0188,
         -0.2206,  0.3095],
        [-0.1245, -0.0296,  0.1414,  0.3304,  0.0785,  0.0095, -0.0416, -0.3064,
         -0.1897, -0.1552]], grad_fn=<AddmmBackward0>)

### 1. 如果将`MySequential`中存储块的方式更改为Python列表，会出现什么样的问题？

In [5]:
class My_sequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.modules1 = []  # 初始化一个空列表用于存储模块
        for block in args:
            self.modules1.append(block)  # 将每个block添加到_module字典中

    def forward(self, X):
        for block in self.modules1:
            X = block(X)
        return X
    
# 实例化My_sequential并传入MLP实例
net = My_sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10)) # 参数传入__init__方法的args中
X = torch.randn(2, 20)  # 生成一个2行20列的随机张量
output = net(X) # X传入forward方法
output # 输出结果

tensor([[ 0.2318, -0.4409,  0.4161,  0.2772,  0.7893,  0.0235,  0.3935, -0.4812,
         -0.2202,  0.2875],
        [ 0.5166, -0.3520,  0.3760, -0.0535,  0.4875,  0.2567,  0.1028, -0.2690,
          0.1677, -0.1506]], grad_fn=<AddmmBackward0>)

#### AttributeError: 'list' object has no attribute 'values'
你的报错原因是：
你在 My_sequential 的 __init__ 方法中用 self.modules1 = [] 存储模块，但在 forward 方法中却写成了 for block in self.modules1.values():，而 list 没有 values() 方法。
#### **注意：**
用列表存储子模块，PyTorch 无法自动注册参数，模型参数***不会被 `optimizer `管理。推荐用 `nn.ModuleList 或 _modules 字典`来存储子模块。***

## 在正向传播函数中执行代码

In [37]:
class FixedHiddenMlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((10, 10), requires_grad=False) # 随机权重，不需要梯度
        self.linear = nn.Linear(10, 10) # 线性层，不需要偏置

    def forward(self, X): # 反向计算是自动求导的
        X = self.linear(X)
        X = F.relu(X)
        # 使用rand_weight进行矩阵乘法
        X = F.relu(torch.mm(X, self.rand_weight))
        # 再次使用线性层
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X

## 混合搭配各种组合块的方法，各个块是可以互相嵌套的

In [38]:
class NestMlp(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 10)
        )
        self.linear = nn.Linear(10, 10)

    def forward(self, X):
        X = self.linear(net(X))
        return X
    
mixednet = nn.Sequential(NestMlp(), FixedHiddenMlp())  # 混合网络
X = torch.randn(2, 20)  # 生成一个2行20列的随机张量
output = mixednet(X)  # 前向传播，输出一个2行10列的张量
# 输出结果
print(output.shape)  # 打印输出的形状，应该是torch.Size([2, 10])

torch.Size([2, 10])


## 练习

1. 如果将`MySequential`中存储块的方式更改为Python列表，会出现什么样的问题？(见顺序块)
1. 实现一个块，它以两个块为参数，例如`net1`和`net2`，并返回前向传播中两个网络的串联输出。这也被称为平行块。
1. 假设我们想要连接同一网络的多个实例。实现一个函数，该函数生成同一个块的多个实例，并在此基础上构建更大的网络。

### 2. 实现一个块，它以两个块为参数，例如`net1`和`net2`，并返回前向传播中两个网络的串联输出。这也被称为平行块。

In [7]:
class ParallelNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net1 = nn.Linear(20, 64)
        self.net2 = nn.Linear(20, 64)

    def forward(self,X):
        out1 = F.relu(self.net1(X))
        out2 = F.relu(self.net2(X))
        return out1, out2  # 返回两个输出
    
# 实例化并测试ParallelNet
net = ParallelNet()
X = torch.randn(2, 20)  # 生成一个2行20列的随机张量
output1, output2 = net(X)  # 前向传播，输出两个张量
# 输出结果
print(output1.shape, output2.shape)  # 打印输出的形状，应该是torch.Size([2, 64]) torch.Size([2, 64])

torch.Size([2, 64]) torch.Size([2, 64])


### 3. 假设我们想要连接同一网络的多个实例。实现一个函数，该函数生成同一个块的多个实例，并在此基础上构建更大的网络。

In [ ]:
def block_n_net(layer_class, num_layers, *args, **kwatgs):
    """
    创建一个包含指定数量层的网络。
    :param layer_class: 要使用的层的类
    :param num_layers: 要创建的层的数量
    :param args: 传递给层的参数
    :param kwatgs: 传递给层的其他参数
    :return: 一个nn.Sequential对象，包含指定数量的层
    """
    layers = []
    for _ in range(num_layers):
        layers.append(layer_class(*args,**kwatgs))  # 添加指定数量的层
        return nn.Sequential(*layers)
# 实例化block_n_net并测试
layer_class = nn.Linear  # 定义一个线性层类
num_layers = 3
net = block_n_net(layer_class, 3, 8, 4) # 创建一个包含3个线性层的网络，每个层的输入维度为8，输出维度为4
X = torch.randn(2, 8)  # 生成一个2行8列的随机张量
output = net(X)  # 前向传播，输出一个2行4列的张量
# 输出结果
print(output.shape)  # 打印输出的形状，应该是torch.Size([2, 4])

torch.Size([2, 4])


# 参数管理

## 首先关注具有单隐藏层的多层感知机

In [39]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(20, 8),
    nn.ReLU(),
    nn.Linear(8, 10)
)

X = torch.randn(2, 20)  # 生成一个2行20列的随机张量
output = net(X)  # 前向传播，输出一个2行10列的张量

### 参数访问

In [40]:
print(net[2].state_dict())  # 打印第二层的状态字典

OrderedDict({'weight': tensor([[ 0.2638, -0.2532,  0.0542, -0.0892, -0.1106, -0.2382,  0.0465, -0.2672],
        [-0.1330, -0.3188,  0.3205, -0.0548,  0.3010,  0.0428, -0.2598,  0.1630],
        [ 0.3001, -0.1747,  0.3220, -0.2940,  0.1141,  0.0885,  0.3260,  0.1711],
        [ 0.2492,  0.2950,  0.2858, -0.1403, -0.2837,  0.1490,  0.0311, -0.2749],
        [-0.0732,  0.2219, -0.2124, -0.3410, -0.2402,  0.2863, -0.0842, -0.2074],
        [ 0.2546,  0.2607,  0.0658,  0.1877,  0.2247,  0.0713,  0.2485,  0.2426],
        [ 0.1461, -0.0956, -0.0889,  0.2450,  0.1169, -0.2942,  0.0598,  0.3039],
        [-0.0133, -0.0765,  0.1231, -0.2079, -0.0970, -0.2553, -0.2606,  0.3531],
        [ 0.1685,  0.0071,  0.3090, -0.0513, -0.0849,  0.0271, -0.1331,  0.1472],
        [ 0.0464,  0.0468, -0.3101,  0.3045,  0.0885, -0.0503, -0.1536,  0.1564]]), 'bias': tensor([-0.1576,  0.1692, -0.0677,  0.2994, -0.0844, -0.0797,  0.1812, -0.0601,
         0.3082,  0.2233])})


### 目标参数

In [41]:
print(net[2].weight)  # 打印第二层的权重
print(net[2].bias)  # 打印第二层的偏置
print(type(net[2].weight))  # 打印第二层权重的类型,应该是torch.nn.parameter.Parameter,Parameter是一个特殊的Tensor，表示模型的参数,可以被优化
print(net[2].bias.data) # 打印第二层偏置数据

net[2].weight.grad == None  # 检查第二层的权重梯度是否为None

Parameter containing:
tensor([[ 0.2638, -0.2532,  0.0542, -0.0892, -0.1106, -0.2382,  0.0465, -0.2672],
        [-0.1330, -0.3188,  0.3205, -0.0548,  0.3010,  0.0428, -0.2598,  0.1630],
        [ 0.3001, -0.1747,  0.3220, -0.2940,  0.1141,  0.0885,  0.3260,  0.1711],
        [ 0.2492,  0.2950,  0.2858, -0.1403, -0.2837,  0.1490,  0.0311, -0.2749],
        [-0.0732,  0.2219, -0.2124, -0.3410, -0.2402,  0.2863, -0.0842, -0.2074],
        [ 0.2546,  0.2607,  0.0658,  0.1877,  0.2247,  0.0713,  0.2485,  0.2426],
        [ 0.1461, -0.0956, -0.0889,  0.2450,  0.1169, -0.2942,  0.0598,  0.3039],
        [-0.0133, -0.0765,  0.1231, -0.2079, -0.0970, -0.2553, -0.2606,  0.3531],
        [ 0.1685,  0.0071,  0.3090, -0.0513, -0.0849,  0.0271, -0.1331,  0.1472],
        [ 0.0464,  0.0468, -0.3101,  0.3045,  0.0885, -0.0503, -0.1536,  0.1564]],
       requires_grad=True)
Parameter containing:
tensor([-0.1576,  0.1692, -0.0677,  0.2994, -0.0844, -0.0797,  0.1812, -0.0601,
         0.3082,  0.2233], r

True

## 一次性访问所有参数

In [42]:
print(*[(name, param.shape)for name, param in net[0].named_parameters()])
print(*[(name, param.shape)for name, param in net.named_parameters()])

('weight', torch.Size([8, 20])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 20])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([10, 8])) ('2.bias', torch.Size([10]))


In [43]:
net.state_dict()['2.bias'].data  # 打印第二层偏置数据

tensor([-0.1576,  0.1692, -0.0677,  0.2994, -0.0844, -0.0797,  0.1812, -0.0601,
         0.3082,  0.2233])

## 从嵌套块中收集参数

In [48]:
def block1():
    return nn.Sequential(
        nn.Linear(4, 8),
        nn.ReLU(),
        nn.Linear(8, 4),
        nn.ReLU()
    )

def block2():
    net = nn.Sequential()
    for i in range(4):
        net.add_module(f'block{i}', block1()) # block2是嵌套的4个block1
    return net
    
rgnet = nn.Sequential(block2(), nn.Linear(4, 1))
X = torch.randn(2, 4)  # 生成一个2行4列的随机张量
rgnet(X)

tensor([[0.1920],
        [0.1920]], grad_fn=<AddmmBackward0>)

In [49]:
print(rgnet)

Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


## 内置初始化

In [ ]:
def init_normal(m):
    if type(m) == nn.Linear: # m是nn.Linear类型的层
        nn.init.normal_(m.weight, mean=0, std=0.01)  # 正态分布初始化权重
        nn.init.zeros_(m.bias)  # 零初始化偏置
net.apply(init_normal)  # 应用初始化函数到网络的每一层  #给你一个神经网络，让你对其进行一个遍历
net[0].weight.data[0], net[0].bias.data[0] # 打印第一层权重数据

(tensor([-0.0046,  0.0213,  0.0076,  0.0037,  0.0110,  0.0018,  0.0194,  0.0030,
         -0.0188,  0.0265,  0.0028,  0.0048,  0.0009, -0.0099,  0.0092, -0.0009,
         -0.0067,  0.0025,  0.0010, -0.0031]),
 tensor(0.))

In [52]:
def init_constant(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)  # 常数初始化权重为1
        nn.init.zeros_(m.bias)  # 常数初始化偏置为0
net.apply(init_constant)  # 应用常数初始化函数到网络的每一层
print(net[0].weight.data[0], net[0].bias.data[0])  # 打印第一层权重和偏置数据 

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1.]) tensor(0.)


### 对某些块应用不同的初始化方法

In [ ]:
def xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)  # Xavier均匀分布初始化权重

def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)  # 常数初始化权重为42

# 对不同的层应用不同的初始化函数
net[0].apply(xavier)  # 应用Xavier初始化函数到第一层
net[2].apply(init_42)  # 应用常数初始化函数到第三层
print(net[0].weight.data[0], net[2].weight.data[0])  # 打印第一层和第三层的权重数据

tensor([-0.1629, -0.3489, -0.1591,  0.1045, -0.0065,  0.2500,  0.3862, -0.0496,
         0.2716,  0.3947,  0.2067,  0.0370, -0.3345, -0.0647,  0.1647,  0.2984,
        -0.1536,  0.4094, -0.3483, -0.3148]) tensor([42., 42., 42., 42., 42., 42., 42., 42.])


### 自定义初始化

In [ ]:
def my_init(m):
    if type(m) == nn.Linear:
        print('Init',*[(name, param.shape) for name, param in m.named_parameters()][0])  # 打印层的参数名称和形状
        nn.init.uniform_(m.weight, -10, 10)  # 均匀分布初始化权重
        m.weight.data *= m.weight.data.abs() >= 5 #大于5的权重保留原数值，小于5的权重置零

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 20])
Init weight torch.Size([10, 8])


tensor([[-0.0000,  0.0000,  5.1351, -5.5493,  7.2570,  0.0000, -0.0000, -0.0000,
          8.8651, -8.2207, -0.0000,  0.0000, -0.0000, -0.0000, -8.3743,  5.0999,
          0.0000,  5.1525,  0.0000, -8.5408],
        [-6.9080,  0.0000, -9.1260, -0.0000, -0.0000,  5.2736, -0.0000, -7.9208,
          0.0000, -6.0767, -0.0000, -0.0000,  8.7809,  0.0000,  0.0000, -6.8490,
          7.7690, -0.0000, -0.0000,  7.0231]], grad_fn=<SliceBackward0>)

### 强制替换

In [54]:
net[0].weight.data[:] += 1
net[0].weight.data[0,0] = 42
net[0].weight.data[0]

tensor([42.0000,  1.0000,  6.1351, -4.5493,  8.2570,  1.0000,  1.0000,  1.0000,
         9.8651, -7.2207,  1.0000,  1.0000,  1.0000,  1.0000, -7.3743,  6.0999,
         1.0000,  6.1525,  1.0000, -7.5408])

## 参数绑定
通常，我们希望在一些层中分享参数，share parameters

In [ ]:
shared = nn.Linear(8, 8) # 实例化后，shared的参数始终一致
net = nn.Sequential(
    nn.Linear(4, 8),nn.ReLU(),
    shared,nn.ReLU(),
    shared,nn.ReLU(),
    nn.Linear(8, 1)
)
X = torch.rand(6, 4)
net(X)
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] == 100
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


这个例子表明第三个和第五个神经网络层的参数是绑定的。
它们不仅值相等，而且由相同的张量表示。

## 练习

1. 如果指定了第一层的输入尺寸，但没有指定后续层的尺寸，会发生什么？是否立即进行初始化？  
    - 只有第一层会立即初始化参数，后续层会在第一次接收到输入时根据输入的形状自动初始化参数（lazy initialization）。如果后续层没有指定输入尺寸，只有在第一次前向传播时才会分配参数。

2. 如果指定了不匹配的维度会发生什么？  
    - 如果输入和层的参数维度不匹配，会在前向传播时报错（通常为`RuntimeError: mat1 and mat2 shapes cannot be multiplied`），提示矩阵乘法的维度不一致。

3. 如果输入具有不同的维度，需要做什么？提示：查看参数绑定的相关内容。  
    - 需要确保所有层的输入输出维度匹配。如果有参数共享（如参数绑定），则共享的层输入输出维度也必须一致。可以通过调整网络结构或使用适当的线性变换（如`nn.Linear`）来适配不同的输入维度。

# 自定义层

## 构造一个没有任何定义的自定义层

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, X):
        return X - X.mean()
    
layer = CenteredLayer()
X = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
output = layer(X)  # 前向传播
output

tensor([-2., -1.,  0.,  1.,  2.])

## 将层作为组建合并到更复杂的模型中

In [10]:
net = nn.Sequential(
    nn.Linear(4, 8),
    CenteredLayer()
)

X = torch.rand(6, 4)  # 生成一个6行4列的随机张量
output = net(X)  # 前向传播，输出一个6行8列的张量
output.mean()

tensor(-2.4835e-09, grad_fn=<MeanBackward0>)

## 带参数的图层

In [11]:
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.zeros(units))

    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        # 使用weight.data和bias.data来获取参数的值
        # 注意：这里使用.data是为了避免梯度计算，通常不推荐这样做
        # 在实际应用中，应该使用self.weight和self.bias来保持梯度计算
        return F.relu(linear)
    
net = MyLinear(4, 6)  # 实例化MyLinear
net.weight

Parameter containing:
tensor([[ 0.8236, -0.6232, -0.8690,  0.2465,  0.2850, -0.5737],
        [-1.8745, -0.0188, -0.0548,  0.2584,  1.4160,  0.1318],
        [-0.2554,  0.5298, -2.0200, -0.0670,  1.1227,  0.1290],
        [-0.4814,  0.8087, -0.9036,  0.3298,  0.9936,  0.6842]],
       requires_grad=True)

### 使用自定义层执行正向传播计算

In [12]:
net(torch.rand(2, 4))  # 前向传播，输出一个2行6列的张量

tensor([[0.0000, 0.2679, 0.0000, 0.5862, 2.9614, 0.0550],
        [0.0000, 0.0000, 0.0000, 0.1487, 0.7978, 0.0000]])

### 使用自定义层构建模型

In [14]:
net = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    MyLinear(8, 6)
)
X = torch.rand(2, 4)  # 生成一个2行4列的随机张量
output = net(X)  # 前向传播，输出一个2行6列的张量
print(output.shape)  # 打印输出的形状，应该是torch.Size([2, 6])
output

torch.Size([2, 6])


tensor([[3.6625, 1.4796, 0.0000, 0.6798, 0.8184, 0.0000],
        [1.8469, 1.0005, 0.0000, 0.1691, 0.3146, 0.0000]],
       grad_fn=<ReluBackward0>)

# 读写文件

## 加载和保存张量

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

x = torch.randn(2, 4)  # 生成一个2行4列的随机张量
torch.save(x, 'x-file.csv') # 保存张量到文件

x2 = torch.load('x-file.csv')  # 从文件加载张量
x2

tensor([[ 0.3088,  1.0910,  0.4450,  0.4187],
        [ 0.5225, -1.3665,  0.3859, -0.7645]])

### 储存一个张量列表，然后把他们读回内存

In [17]:
y = torch.randn(2, 4)  # 生成另一个2行4列的随机张量
torch.save([x, y], 'xy-file.csv')  # 保存多个张量到文件
x2, y2 = torch.load('xy-file.csv')  # 从文件加载多个张量
x2, y2

(tensor([[ 0.3088,  1.0910,  0.4450,  0.4187],
         [ 0.5225, -1.3665,  0.3859, -0.7645]]),
 tensor([[ 0.7965, -0.2833, -0.4324, -0.2046],
         [ 0.2840, -0.4400, -2.4649, -0.9007]]))

### 写入或读取从字符串映射到张量的字典

In [18]:
mydict = {'x': x, 'y': y}  # 创建一个字典
torch.save(mydict, 'mydict-file.csv')  # 保存字典到文件
mydict2 = torch.load('mydict-file.csv')  # 从文件加载字典
mydict2

{'x': tensor([[ 0.3088,  1.0910,  0.4450,  0.4187],
         [ 0.5225, -1.3665,  0.3859, -0.7645]]),
 'y': tensor([[ 0.7965, -0.2833, -0.4324, -0.2046],
         [ 0.2840, -0.4400, -2.4649, -0.9007]])}

## 加载和保存模型参数

In [19]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))
    
net = MLP()  # 实例化网络
X = torch.randn(2, 20)  # 生成一个2行20列的随机张量
output = net(X)  # 前向传播，输出一个2行10列的张量

### 把模型储存为一个叫做mlp.params的文件

In [20]:
torch.save(net.state_dict(), 'mlp-params')  # 保存模型参数到文件

### 实例化原始多层感知机模型的一个备份，直接读取文件中储存的参数

In [21]:
clone = MLP()  # 克隆一个新的网络实例
clone.load_state_dict(torch.load('mlp-params'))  # 从文件加载模型参数到克隆网络 
clone.eval()  # 设置克隆网络为评估模式

MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (out): Linear(in_features=256, out_features=10, bias=True)
)

In [22]:
y_clone = clone(X)  # 使用克隆网络进行前向传播
y_clone == output  # 检查克隆网络的输出是否与原网络相同

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

## 练习

1. 即使不需要将经过训练的模型部署到不同的设备上，存储模型参数还有什么实际的好处？
    - 可以在训练中断后恢复训练进度，便于实验复现和调试；也可以用于模型版本管理，方便对比不同实验结果。

2. 假设我们只想复用网络的一部分，以将其合并到不同的网络架构中。比如想在一个新的网络中使用之前网络的前两层，该怎么做？
    - 可以通过`nn.Sequential`或直接访问原网络的子模块（如`net[:2]`），将其作为新网络的子模块。例如：`new_net = nn.Sequential(net[0], net[1], ...)`。

3. 如何同时保存网络架构和参数？需要对架构加上什么限制？
    - 可以使用`torch.save(model, path)`直接保存整个模型（包括架构和参数），但要求模型的定义在加载时可用（即类定义必须可导入且一致）。推荐方式是保存参数（`state_dict`）和代码分开管理，保证可移植性和灵活性。